# 01 — Stage A: Problem Framing and Metric Selection

**Steps 1–2 of the 11-step predictive-modeling process.**

The process calls this stage *"before touching code"*, and that is meant literally. This
notebook records **decisions**, not computation: what problem is being solved, for whom,
what an error costs, and which number will be used to declare success. Everything in
notebooks 02–04 is downstream of the choices made here.

The two code cells below exist only to make a decision concrete (the cost matrix) and to
demonstrate why one metric was rejected (accuracy). No project data is loaded.

| Notebook | Stage | Steps |
|---|---|---|
| **01 (this one)** | **A — before code** | **1. Problem framing · 2. Type, target, metric** |
| 02 | B — data | 3. Understand · 4. Split · 5. EDA · 6. Clean · 7. Features |
| 03 | C — model | 8. Build · 9. Evaluate · 10. Explain |
| 04 | D — production | 11. Package, deploy, monitor |

> The process is a **loop, not a line**. Results downstream regularly force a return to
> this notebook — in particular, the cost ratio in Step 1 should be revisited once the
> segment and fairness analysis in notebook 03 shows who actually bears the errors.

## Step 1 — Understand the business problem

### 1.0 Problem summary

**The domain.** Lending Club was a US **peer-to-peer (P2P) lending** marketplace. Borrowers
applied online for unsecured personal loans of roughly \$1,000–\$40,000 over a 36- or
60-month term; investors funded those loans in small increments and received the
repayments. Lending Club earned origination fees and **did not itself carry the credit
risk** — when a borrower stops paying, the loss falls on the investor. That is what makes
an accurate default estimate valuable to somebody, which is the first thing worth
establishing about any prediction problem.

The public loan book for 2007–2018 Q4 is the source dataset. The modelling scope is the
**36-month loans issued 2012–2015** — the vintages that had all reached maturity by the
2018 Q4 snapshot, so every outcome is known (see difficulty 1 below).

**The task.** *Credit scoring*, and specifically **probability-of-default (PD)
estimation**: at the moment an application arrives, estimate how likely that borrower is
to stop repaying before the term ends. This is one of the oldest and most heavily
regulated applications of statistical learning in finance, which is why the conventions
adopted later in this project — scorecards, KS and Gini, calibration, population
stability — come from banking practice rather than from general machine learning.

**Why it is hard.** Three structural difficulties, all of which shape later decisions:

1. **The label arrives years late.** A 60-month loan issued in 2018 is not resolved until
   2023. Only loans that have reached a terminal state can be labelled at all, and for
   recent vintages the ones that *have* resolved are a biased subsample (early defaults
   and early prepayments). We therefore restrict the scope to vintages whose scheduled
   term had fully elapsed by the snapshot — 36-month loans issued 2012–2015.
2. **The signal is weak and diffuse.** Default is driven substantially by events *after*
   origination — job loss, illness, divorce — that no application form observes. Published
   work on this dataset lands at ROC-AUC ≈ 0.68–0.72 (`docs/related_work.md`); that is the
   realistic ceiling, not a sign of a poor model.
3. **Errors are asymmetric and the classes imbalanced.** Roughly 1 loan in 5 defaults, and
   the two mistakes cost very different amounts — see §1.3.

#### Input and output

| | |
|---|---|
| **One row** | One **funded loan application**, at the moment the decision is made |
| **Input** | ≈30 application-time fields: loan request (amount, term, purpose), applicant profile (income, employment, housing, state), and credit-bureau summary (FICO band, DTI, utilisation, open accounts, delinquencies, credit-history length) |
| **Output** | A single **calibrated probability** in `[0, 1]` that the loan ends in `Charged Off` / `Default` |
| **Decision** | That probability compared against a cut-off derived from the cost matrix in §1.3 → approve / decline, price tier, exposure |
| **Prediction time** | At application, **before** funding — no post-origination field may be used |
| **Outcome horizon** | The full loan term — 36 months in the modelling scope |

**Explicitly out of scope.** Loss Given Default and Exposure at Default (so no full
expected-loss figure), rejected applicants (they carry no outcome — see the selection-bias
limitation in notebook 03), fraud detection, and collections or recovery strategy after
default.

**In one sentence.** *Given the ~30 fields a lender holds when an application arrives,
estimate the probability that the loan will be charged off — well enough calibrated to
price and decide on, and better than the grade Lending Club already assigns.*

The rest of Step 1 makes each part of that sentence precise: who decides (§1.1), at what
moment (§1.2), what being wrong costs (§1.3), and what "better than" is measured against
(§1.4).

### 1.1 Who uses the prediction, and what decision changes?

A credit-risk function at a consumer lending platform. The score changes three decisions:

| Decision | How the score is used |
|---|---|
| **Approve / decline** | Applications above a risk cut-off are declined |
| **Price** | Risk tier sets the interest rate offered |
| **Exposure** | Riskier approved applicants are offered a lower amount or shorter term |

If none of these decisions would change, the prediction has no value and the project
should stop. They do change, so it continues.

### 1.2 What is predicted, and at what moment?

**`P(the loan is charged off before its term ends)`, evaluated at the moment of
application**, using only information a lender holds at that moment.

The timing half of that sentence is the load-bearing part. It is what makes
`recoveries`, `total_pymnt` and `last_fico_range_*` inadmissible — they exist only after
money has been lent — and it is the test every candidate feature must pass in Step 3.

### 1.3 What do errors cost?

The two errors are **not symmetric**, which is the single most consequential fact about
this problem:

| Error | What happens | Rough magnitude |
|---|---|---|
| **False negative** — predicted repay, actually defaults | The loan is approved and the principal is largely lost | Order of the loan amount |
| **False positive** — predicted default, actually repays | A good applicant is declined; the interest margin is foregone | Order of the margin on one loan |

A missed default costs several times a wrongly declined applicant. This asymmetry, not
the choice of algorithm, is what should drive the decision threshold — and it is why
accuracy is disqualified as a headline metric in Step 2.

### 1.4 What is the current approach? *(the baseline to beat)*

**Lending Club already assigns every application a `grade`, a `sub_grade` and an
`int_rate`.** That is an existing, working risk model, and it is the incumbent this
project must beat.

This matters more than it first appears:

- Those columns are legitimately available at application time, so they may be used as
  features — but a model that leans on them is substantially **re-learning the
  incumbent's model** rather than adding information.
- It gives a hard, non-trivial baseline: rank applicants by `sub_grade` alone. Beating
  the majority-class baseline is easy; beating `sub_grade` is the real test.
- Both baselines are specified in Step 2 and **computed in notebook 03, Step 8**.

### 1.5 Does this need machine learning at all?

Worth asking honestly. A monotone cut-off on `sub_grade` is already a functioning
decision rule requiring no model at all. Machine learning earns its place here only if it
adds measurable lift over that rule, and the report should say so plainly if it does not.
The complexity is not free: an ensemble is harder to audit, harder to explain to a
declined applicant, and harder to monitor.

In [ ]:
# The cost matrix from 1.3, as a decision recorded in code.
# These are ILLUSTRATIVE placeholders. Replace with real portfolio economics before
# quoting any threshold-dependent number (precision, recall, F1, cost per loan).
import pandas as pd

COST_FN = 1.00   # missed default: unrecovered principal, normalised to 1
COST_FP = 0.25   # rejected good applicant: foregone interest margin

cost_matrix = pd.DataFrame(
    [[0.0, COST_FP], [COST_FN, 0.0]],
    index=["Actual: Fully Paid", "Actual: Charged Off"],
    columns=["Predicted: Fully Paid", "Predicted: Charged Off"])

print(f"Cost ratio FN:FP = {COST_FN / COST_FP:.0f}:1\n")
display(cost_matrix.style.format("{:.2f}").set_caption(
    "Table 1 - Cost matrix (illustrative units, normalised to the cost of one missed default)"))

**Table 1 — The cost matrix.**

A 4:1 ratio says the lender should accept roughly four wrongly declined good applicants to
avoid one missed default. Notebook 03 turns this directly into a decision threshold, and
because the ratio is a *policy input* rather than a model output, changing it changes the
operating point without retraining anything.

**Action →** Carry `COST_FN` / `COST_FP` into notebook 03 Step 9 as the cost-based
threshold rule, and re-derive the threshold rather than defaulting to 0.5.

## Step 2 — Problem type, target variable and metric

### 2.1 Problem type and the split that goes with it

| Problem type | Metrics | Split |
|---|---|---|
| Regression | RMSE, MAE, MAPE, R² | Random |
| Balanced classification | Accuracy, F1 | Stratified |
| **Imbalanced classification** | **Precision/Recall, PR-AUC, F-beta** | Stratified |
| **Time series / forecasting** | MAE, MASE, sMAPE | **By time** |
| Ranking | NDCG, MAP | By user or time |

This project is **imbalanced binary classification** (~20 % positive) — so the third row
supplies the metrics.

**But we take the split from the fourth row, not the third, and that is deliberate.**
The reference table pairs imbalanced classification with a stratified split; we use an
**out-of-time** split instead. The reason is the deployment question from Step 1.2: the
model scores *future* applicants, so the evaluation has to be a forecast. Applicants
arrive in time order, and the platform's credit policy, product mix and macroeconomic
backdrop all drift across the window. A random split would let the model train on 2015
loans in order to predict 2013 ones — information no deployed scorecard could have.

This is a considered deviation, not an oversight. Notebook 03 quantifies exactly how much
optimism the random split would have bought.

### 2.2 The target variable

A credit label requires a **resolved** outcome:

| `loan_status` | Target | Why |
|---|---|---|
| `Fully Paid` | `0` | Resolved, repaid |
| `Charged Off`, `Default` | `1` | Resolved, defaulted |
| `Current`, `Late …`, `In Grace Period`, `Issued` | *excluded* | Outcome not yet known |

Excluding unresolved loans is unavoidable, and on recent vintages it is not free: the
loans that have resolved early are a biased subsample (right-censoring). We avoid that cost
by scope rather than by caveat — only **36-month loans issued 2012–2015** are used, all of
which had passed their scheduled maturity by the 2018 Q4 snapshot, so almost every one is
resolved. The price is a narrower population (no 60-month loans, nothing after 2015),
recorded in notebook 03's limitations.

### 2.3 Metric selection — driven by the Step 1 costs

| Metric | Role | Why this role |
|---|---|---|
| **PR-AUC** (average precision) | **Primary** | Focuses on the rare, expensive positive class. Its baseline is the prevalence (~0.20), so it cannot be inflated by the majority class |
| **Brier score** | **Co-primary** | The probability *is* the product — expected loss is `PD × EAD × LGD`, and pricing consumes the number itself. The only metric here that punishes miscalibration |
| ROC-AUC | Secondary | Threshold-free ranking quality; conventional and comparable, but optimistic under imbalance because true negatives dominate both rates |
| KS | Secondary | Industry-standard separation measure in credit scoring |
| Gini (`2·AUC − 1`) | Secondary | Rescaled ROC-AUC; reported because credit risk conventionally does |
| Precision / Recall / F1 | At the operating point | Meaningful only once the threshold is justified by the cost matrix |
| **Accuracy** | **Reported to be dismissed** | See below |

### 2.4 Why accuracy is disqualified

The classic imbalanced-data trap: at 20 % prevalence, a model that predicts "nobody ever
defaults" achieves 80 % accuracy while being completely useless. The cell below
demonstrates it rather than asserting it.

In [ ]:
import numpy as np
from sklearn.metrics import (accuracy_score, average_precision_score, f1_score,
                             roc_auc_score)

rng = np.random.default_rng(42)
PREVALENCE = 0.20
n = 50_000

y_demo = (rng.random(n) < PREVALENCE).astype(int)

demo = pd.DataFrame([
    {"Strategy": "Always predict 'Fully Paid'",
     "Accuracy": accuracy_score(y_demo, np.zeros(n)),
     "Recall (defaults caught)": 0.0,
     "F1": f1_score(y_demo, np.zeros(n), zero_division=0),
     "ROC-AUC": 0.5, "PR-AUC": PREVALENCE},
    {"Strategy": "Random guessing",
     "Accuracy": accuracy_score(y_demo, (rng.random(n) < PREVALENCE).astype(int)),
     "Recall (defaults caught)": PREVALENCE,
     "F1": f1_score(y_demo, (rng.random(n) < PREVALENCE).astype(int), zero_division=0),
     "ROC-AUC": 0.5, "PR-AUC": PREVALENCE},
]).set_index("Strategy")

display(demo.style.format("{:.3f}").set_caption(
    "Table 2 - Degenerate strategies at 20% prevalence: accuracy flatters, PR-AUC does not"))

always_repay = demo.iloc[0]
print(f"A model that never predicts a default is {always_repay['Accuracy']:.1%} accurate,")
print(f"and catches {always_repay['Recall (defaults caught)']:.0%} of defaults.")

**Table 2 — Degenerate strategies evaluated at 20 % prevalence.**

The "always predict Fully Paid" row scores 80 % accuracy while catching zero defaults —
the exact failure the process warns about. PR-AUC pins both degenerate strategies at the
prevalence (0.20), which is why it is the primary metric: a model must clear that floor to
have demonstrated anything at all.

Note this also sets the bar for reading notebook 03's results. A PR-AUC of 0.29 is not
"bad because it is far from 1.0" — it is **45 % above the 0.20 floor**, and that lift is
the quantity of interest.

**Action →** In notebook 03, report every model against the prevalence floor, and keep
accuracy in the table solely as evidence that it fails to discriminate between a useful
model and a useless one.

### 2.5 Baselines to beat

> *Always compute a baseline.*

Two, in increasing order of difficulty. Both are **computed in notebook 03, Step 8**,
where the train/test split exists; they are specified here so the bar is set before any
modelling result is seen.

| # | Baseline | What it represents | Expected |
|---|---|---|---|
| **B0** | **Majority class** (`DummyClassifier(strategy="prior")`) | The degenerate model from Table 2 | ROC-AUC 0.50, PR-AUC ≈ prevalence |
| **B1** | **`sub_grade` as a risk score** | *The incumbent from Step 1.4* — Lending Club's existing underwriting | Substantially better than chance |

**B1 is the one that matters.** Beating B0 proves nothing. A model that cannot beat B1 has
not justified its existence over the rule the platform already uses, and the report should
say exactly that if it happens.

---

## Stage A decisions, carried forward

| Decision | Value | Used in |
|---|---|---|
| Prediction target | `P(charged off)` at application time | NB02 Step 3 (timing test) |
| Target encoding rule | Terminal statuses only | NB02 Step 3 |
| Cost ratio | `FN : FP = 4 : 1` | NB03 Step 9 (threshold) |
| Primary metric | PR-AUC, with Brier co-primary | NB03 Step 9 |
| Split protocol | Out-of-time, deliberately not stratified | NB02 Step 4 |
| Baselines | B0 majority class, B1 `sub_grade` | NB03 Step 8 |

**Next:** `02_data_and_features.ipynb` — Steps 3–7.